© 2026 Mohith Gowda V. All rights reserved.

This notebook and its contents are the intellectual property of Mohith Gowda.
Unauthorized copying, modification, distribution, or use of this material,
in whole or in part, without explicit permission is strictly prohibited.

Created for educational, professional, and interview preparation purposes.


In [0]:
base_path = "/Volumes/dev/spark_db/datasets/Storedataset"

customers_df = spark.read.option("header", True).option("inferSchema", True) \
    .csv(f"{base_path}/customers.csv")

departments_df = spark.read.option("header", True).option("inferSchema", True) \
    .csv(f"{base_path}/departments.csv")

employees_df = spark.read.option("header", True).option("inferSchema", True) \
    .csv(f"{base_path}/employees.csv")

orders_df = spark.read.option("header", True).option("inferSchema", True) \
    .csv(f"{base_path}/orders.csv")

order_items_df = spark.read.option("header", True).option("inferSchema", True) \
    .csv(f"{base_path}/order_items.csv")

products_df = spark.read.option("header", True).option("inferSchema", True) \
    .csv(f"{base_path}/products.csv")

returns_df = spark.read.option("header", True).option("inferSchema", True) \
    .csv(f"{base_path}/returns.csv")

web_logs_df = spark.read.option("header", True).option("inferSchema", True) \
    .csv(f"{base_path}/web_logs.csv")


In [0]:
customers_df.createOrReplaceTempView("customers")
departments_df.createOrReplaceTempView("departments")
employees_df.createOrReplaceTempView("employees")
orders_df.createOrReplaceTempView("orders")
order_items_df.createOrReplaceTempView("order_items")
products_df.createOrReplaceTempView("products")
returns_df.createOrReplaceTempView("returns")
web_logs_df.createOrReplaceTempView("web_logs")


In [0]:

from pyspark.sql.functions import col, sum, rank , round , avg
from pyspark.sql.window import Window


## Q1. For each product category, list the top 3 products by total revenue from delivered orders only.

```sql
SELECT category, product_name, revenue
FROM (
  SELECT p.category, p.product_name, SUM(oi.line_total) AS revenue,
         RANK() OVER (PARTITION BY p.category ORDER BY SUM(oi.line_total) DESC) rk
  FROM order_items oi
  JOIN orders o    ON oi.order_id = o.order_id
  JOIN products p  ON oi.product_id = p.product_id
  WHERE o.status = 'Delivered'
  GROUP BY p.category, p.product_name
) t WHERE rk <= 3;
``

In [0]:

revenue_df = (
    order_items_df.alias("oi")
    .join(orders_df.alias("o"), col("oi.order_id") == col("o.order_id"))
    .join(products_df.alias("p"), col("oi.product_id") == col("p.product_id"))
    .where(col("o.status") == "Delivered")
    .groupBy("p.category", "p.product_name")
    .agg(sum("oi.line_total").alias("revenue")))



ranked_df = revenue_df.withColumn("rk", rank().over(Window.partitionBy("category").orderBy(col("revenue").desc())))



final_df = ranked_df.filter(col("rk") <= 3) \
                    .select("category", "product_name", "revenue")

final_df_formatted = final_df.withColumn(
    "revenue",
    col("revenue").cast("decimal(18,6)")
)

final_df_formatted.show(truncate=False)

+-----------+---------------------+---------------+
|category   |product_name         |revenue        |
+-----------+---------------------+---------------+
|Books      |Fiction_Model_56     |11003232.890000|
|Books      |Textbook_Model_71    |10780328.870000|
|Books      |Fiction_Model_57     |10519083.030000|
|Clothing   |Dress_Model_27       |11907640.340000|
|Clothing   |Saree_Model_33       |11156594.970000|
|Clothing   |Shoes_Model_30       |10332208.100000|
|Electronics|Laptop_Model_4       |14747809.790000|
|Electronics|Camera_Model_12      |10333796.120000|
|Electronics|Camera_Model_14      |9547137.590000 |
|Grocery    |Snacks_Model_85      |8450792.210000 |
|Grocery    |Flour_Model_75       |8100914.300000 |
|Grocery    |Spices_Model_83      |7824961.620000 |
|Home       |Dining Table_Model_50|13201954.310000|
|Home       |Dining Table_Model_51|11986403.890000|
|Home       |Carpet_Model_46      |11285316.930000|
+-----------+---------------------+---------------+



## Q2. Find customers whose total spend in the "Electronics" category is greater than the average spend of all customers in that same category.

```sql
WITH spend AS (
  SELECT o.customer_id, SUM(oi.line_total) tot
  FROM orders o JOIN order_items oi ON o.order_id=oi.order_id
                JOIN products p    ON oi.product_id=p.product_id
  WHERE p.category='Electronics'
  GROUP BY o.customer_id
)
SELECT customer_id, tot FROM spend
WHERE tot > (SELECT AVG(tot) FROM spend);
``

In [0]:
spend_df = (order_items_df.alias("oi")
    .join(orders_df.alias("o"), col("oi.order_id") == col("o.order_id"))
    .join(products_df.alias("p"), col("oi.product_id") == col("p.product_id"))
    .where(col("p.category") == "Electronics")
    .groupBy("o.customer_id")
    .agg(sum("oi.line_total").alias("tot")))

avg_spend = spend_df.select(avg("tot").alias("avg_tot")).collect()[0][0]



final_df = spend_df.filter(col("tot") > avg_spend)

final_df.show()


+-----------+------------------+
|customer_id|               tot|
+-----------+------------------+
|         70|         413722.19|
|        471|         420010.08|
|         67|         396025.07|
|        475|          519009.7|
|        257|430470.33999999997|
|        412|         829950.21|
|        275| 589955.0700000001|
|        343|         523205.72|
|        300|487071.93999999994|
|        295|         372375.18|
|        225|         505687.83|
|        186|401937.81999999995|
|        433|         399449.63|
|        375|          482371.3|
|        414|         759646.24|
|        146| 838798.3800000001|
|        151|519828.95999999996|
|        338| 840711.5900000001|
|        432|         704008.64|
|        428| 526723.3799999999|
+-----------+------------------+
only showing top 20 rows


## Q3. Produce an enriched order-line view: customer_name, customer_city, product_name, category, quantity, line_total for all orders placed in 2024 by Premium or Platinum customers.

```sql
SELECT
    c.customer_name,
    c.customer_city,
    p.product_name,
    p.category,
    oi.quantity,
    oi.line_total
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
JOIN products p
    ON oi.product_id = p.product_id
WHERE
    YEAR(o.order_date) = 2024
    AND c.segment IN ('Premium', 'Platinum');
``

In [0]:
from pyspark.sql.functions import col, year

enriched_orders_df = (
    orders_df.alias("o")
    .join(customers_df.alias("c"),
          col("o.customer_id") == col("c.customer_id"))
    .join(order_items_df.alias("oi"),
          col("o.order_id") == col("oi.order_id"))
    .join(products_df.alias("p"),
          col("oi.product_id") == col("p.product_id"))
    .filter(
        (year(col("o.order_date")) == 2024) &
        (col("c.segment").isin("Premium", "Platinum"))
    )
    .select(
        col("c.name"),
        col("c.city"),
        col("p.product_name"),
        col("p.category"),
        col("oi.quantity"),
        col("oi.line_total")
    )
)

enriched_orders_df.show()

+--------+---------+--------------------+-----------+--------+----------+
|    name|     city|        product_name|   category|quantity|line_total|
+--------+---------+--------------------+-----------+--------+----------+
|Cust_496|Bengaluru|      Laptop_Model_2|Electronics|       1|  19220.87|
| Cust_59|  Lucknow|      Sugar_Model_78|    Grocery|       4| 166849.24|
|Cust_230|Ahmedabad|      Saree_Model_33|   Clothing|       3| 202234.35|
|Cust_152|  Lucknow|      Shirt_Model_19|   Clothing|       1|   7933.73|
|Cust_179|  Lucknow|  Biography_Model_67|      Books|       4|  26224.71|
|Cust_220|Hyderabad|Non-Fiction_Model_59|      Books|       4|  106287.2|
|Cust_306|  Lucknow|     Spices_Model_83|    Grocery|       4| 190099.28|
|Cust_433|  Chennai|Non-Fiction_Model_58|      Books|       3| 136072.02|
|Cust_471|  Chennai|       Rice_Model_72|    Grocery|       2|  20223.49|
|Cust_494|  Lucknow|       Sofa_Model_38|       Home|       1|  49120.91|
|Cust_213|   Mumbai|      Sugar_Model_

## Q4. For each day in 2024, compute the daily revenue AND the cumulative revenue from Jan 1 up to that day.

WITH daily_revenue AS (
    SELECT
        CAST(o.order_date AS DATE) AS order_date,
        SUM(oi.line_total) AS daily_revenue
    FROM orders o
    JOIN order_items oi
        ON o.order_id = oi.order_id
    WHERE YEAR(o.order_date) = 2024
    GROUP BY CAST(o.order_date AS DATE)
)
SELECT
    order_date,
    daily_revenue,
    SUM(daily_revenue) OVER (
        ORDER BY order_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_revenue
FROM daily_revenue
ORDER BY order_date;
``

In [0]:
from pyspark.sql.functions import col, sum, year, to_date
from pyspark.sql.window import Window

daily_df = (
    orders_df.alias("o")
    .join(order_items_df.alias("oi"),
          col("o.order_id") == col("oi.order_id"))
    .filter(year(col("o.order_date")) == 2024)
    .groupBy(to_date(col("o.order_date")).alias("order_date"))
    .agg(sum("oi.line_total").alias("daily_revenue"))
)

window_spec = Window.orderBy("order_date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

final_df = (
    daily_df
    .withColumn(
        "cumulative_revenue",
        sum("daily_revenue").over(window_spec)
    )
    .orderBy("order_date")
)

final_df.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+------------------+--------------------+
|order_date|     daily_revenue|  cumulative_revenue|
+----------+------------------+--------------------+
|2024-01-01|        1616696.16|          1616696.16|
|2024-01-02|        1476555.29|          3093251.45|
|2024-01-03|519826.11000000004|          3613077.56|
|2024-01-04|         719778.28|          4332855.84|
|2024-01-05|         265685.24|          4598541.08|
|2024-01-06|          47524.82|           4646065.9|
|2024-01-09|         159268.61|   4805334.510000001|
|2024-01-10| 882824.7300000001|   5688159.240000001|
|2024-01-12|        1099580.28|   6787739.520000001|
|2024-01-13|         1303954.5|   8091694.020000001|
|2024-01-14| 889582.4899999999|   8981276.510000002|
|2024-01-15|         575892.39|   9557168.900000002|
|2024-01-16|         303379.45|   9860548.350000001|
|2024-01-17|1964369.9200000004|1.1824918270000001E7|
|2024-01-18| 649534.2799999999|       1.247445255E7|
|2024-01-19|        1070434.16|       1.354488

## Q5. Group customers into signup-month cohorts. For each cohort, count how many were still active (placed ≥ 1 order) in each of the subsequent 6 months.

WITH cohorts AS (
    SELECT
        customer_id,
        DATE_TRUNC('month', signup_date) AS signup_month
    FROM customers
),

orders_with_month AS (
    SELECT
        o.customer_id,
        DATE_TRUNC('month', o.order_date) AS order_month
    FROM orders o
),

activity AS (
    SELECT
        c.signup_month,
        MONTHS_BETWEEN(o.order_month, c.signup_month) AS month_offset,
        c.customer_id
    FROM cohorts c
    JOIN orders_with_month o
        ON c.customer_id = o.customer_id
    WHERE MONTHS_BETWEEN(o.order_month, c.signup_month) BETWEEN 0 AND 6
)

SELECT
    signup_month,
    CAST(month_offset AS INT) AS month_offset,
    COUNT(DISTINCT customer_id) AS active_customers
FROM activity
GROUP BY signup_month, month_offset
ORDER BY signup_month, month_offset;
``

In [0]:
from pyspark.sql.functions import col, date_trunc, months_between

cohorts_df = customers_df.select(
    col("customer_id"),
    date_trunc("month", col("signup_date")).alias("signup_month")
)

orders_month_df = orders_df.select(
    col("customer_id"),
    date_trunc("month", col("order_date")).alias("order_month")
)

activity_df = (
    cohorts_df
    .join(orders_month_df, "customer_id")
    .withColumn(
        "month_offset",
        months_between(col("order_month"), col("signup_month"))
    )
    .filter(col("month_offset").between(0, 6))
)

result_df = (
    activity_df
    .groupBy(
        "signup_month",
        col("month_offset").cast("int")
    )
    .count()
    .withColumnRenamed("count", "active_customers")
    .orderBy("signup_month", "month_offset")
)

result_df.show()

+-------------------+------------+----------------+
|       signup_month|month_offset|active_customers|
+-------------------+------------+----------------+
|2022-08-01 00:00:00|           5|               2|
|2022-08-01 00:00:00|           6|               2|
|2022-10-01 00:00:00|           3|               6|
|2022-10-01 00:00:00|           4|               1|
|2022-10-01 00:00:00|           6|               2|
|2022-11-01 00:00:00|           2|               2|
|2022-11-01 00:00:00|           3|               1|
|2022-11-01 00:00:00|           4|               2|
|2022-11-01 00:00:00|           5|               1|
|2022-11-01 00:00:00|           6|               1|
|2022-12-01 00:00:00|           1|               1|
|2022-12-01 00:00:00|           2|               1|
|2022-12-01 00:00:00|           3|               2|
|2022-12-01 00:00:00|           4|               1|
|2022-12-01 00:00:00|           5|               2|
|2022-12-01 00:00:00|           6|               1|
|2023-01-01 

## Q6. For each employee, list the full chain of managers up to the CEO (emp_id → [mgr1, mgr2, ..., CEO]) and the depth in the org tree.

WITH RECURSIVE tree AS (
  SELECT emp_id, manager_id, 0 AS depth FROM employees WHERE manager_id IS NULL
  UNION ALL
  SELECT e.emp_id, e.manager_id, t.depth+1
  FROM employees e JOIN tree t ON e.manager_id = t.emp_id
) SELECT * FROM tree;

In [0]:
from pyspark.sql.functions import lit

tree_df = (
    employees_df
    .filter("manager_id IS NULL")
    .select("emp_id", "manager_id")
    .withColumn("depth", lit(0))
)

current_level = tree_df
max_depth = 10  

for i in range(max_depth):
    next_level = (
        employees_df.alias("e")
        .join(current_level.alias("t"),
              col("e.manager_id") == col("t.emp_id"),
              "inner")
        .select(
            col("e.emp_id"),
            col("e.manager_id"),
            (col("t.depth") + 1).alias("depth")
        )
    )

    if next_level.count() == 0:
        break

    tree_df = tree_df.union(next_level)
    current_level = next_level

tree_df.orderBy("depth", "emp_id").show()

+------+----------+-----+
|emp_id|manager_id|depth|
+------+----------+-----+
|     1|      NULL|    0|
|     2|         1|    1|
|     3|         1|    1|
|     4|         1|    1|
|     5|         1|    1|
|     6|         5|    2|
|     7|         4|    2|
|     8|         3|    2|
|     9|         2|    2|
|    10|         2|    2|
|    11|         4|    2|
|    12|         3|    2|
|    13|         5|    2|
|    14|         3|    2|
|    15|         4|    2|
|    16|         4|    2|
|    17|         4|    2|
|    18|         2|    2|
|    19|         4|    2|
|    20|         2|    2|
+------+----------+-----+
only showing top 20 rows


## Q7. For each department, find the employee with the 2nd highest salary (not the max). If a department has < 2 employees, exclude it.

SELECT department, emp_name, salary
FROM (
    SELECT
        department,
        emp_name,
        salary,
        DENSE_RANK() OVER (
            PARTITION BY department
            ORDER BY salary DESC
        ) rk
    FROM employees
) t
WHERE rk = 2;
``

In [0]:


from pyspark.sql.functions import col, dense_rank
from pyspark.sql.window import Window

window_spec = (
    Window
    .partitionBy("department")
    .orderBy(col("salary").desc())
)

result_df = (
    employees_df
    .withColumn("rk", dense_rank().over(window_spec))
    .filter(col("rk") == 2)
    .select(
        col("department"),
        col("emp_name"),
        col("salary")
    )
)

result_df.show()

+-----------+--------+------+
| department|emp_name|salary|
+-----------+--------+------+
|Engineering| Emp_175| 93540|
|    Finance|  Mgr_16|128806|
|         HR|   Mgr_9|119535|
|  Marketing|  Mgr_18|124654|
| Operations|  Mgr_17|113837|
|      Sales| Emp_168| 88464|
|    Support| Emp_102| 92020|
+-----------+--------+------+



## Q8. Find customers who have purchased at least one product in "Electronics" but have never purchased anything in "Clothing".

SELECT DISTINCT o.customer_id
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
WHERE p.category = Electronics

EXCEPT

SELECT DISTINCT o.customer_id
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
WHERE p.category = Clothing;


In [0]:
electronics_df = (
    orders_df
    .join(order_items_df, orders_df.order_id == order_items_df.order_id)
    .join(products_df, order_items_df.product_id == products_df.product_id)
    .filter(products_df.category == "Electronics")
    .select(orders_df.customer_id)
    .distinct()
)

clothing_df = (
    orders_df
    .join(order_items_df, orders_df.order_id == order_items_df.order_id)
    .join(products_df, order_items_df.product_id == products_df.product_id)
    .filter(products_df.category == "Clothing")
    .select(orders_df.customer_id)
    .distinct()
)

result_df = electronics_df.subtract(clothing_df)

result_df.show()


+-----------+
|customer_id|
+-----------+
|        295|
|        138|
|          2|
|         28|
|        411|
|        364|
|        336|
|        317|
|        213|
|         81|
|        380|
|        461|
|        484|
|        373|
|        468|
|        335|
|        283|
|        155|
|        236|
|        237|
+-----------+
only showing top 20 rows


##Q9. A session is a sequence of events by the same customer where no two consecutive events are more than 30 minutes apart. Assign a session_id to every log row and report: total sessions per customer, avg session length in minutes, avg pages per session.


WITH ordered AS (
    SELECT
        customer_id,
        event_time,
        LAG(event_time) OVER (
            PARTITION BY customer_id
            ORDER BY event_time
        ) AS prev_event
    FROM web_logs
),
sessions AS (
    SELECT
        customer_id,
        event_time,
        CASE
            WHEN prev_event IS NULL
              OR TIMESTAMPDIFF(minute, prev_event, event_time) > 30
            THEN 1
            ELSE 0
        END AS new_session
    FROM ordered
),
session_ids AS (
    SELECT
        customer_id,
        event_time,
        SUM(new_session) OVER (
            PARTITION BY customer_id
            ORDER BY event_time
        ) AS session_id
    FROM sessions
)
SELECT
    customer_id,
    COUNT(DISTINCT session_id) AS total_sessions,
    AVG(session_duration) AS avg_session_minutes,
    AVG(page_count) AS avg_pages_per_session
FROM (
    SELECT
        customer_id,
        session_id,
        COUNT(*) AS page_count,
        (MAX(event_time) - MIN(event_time)) / 60 AS session_duration
    FROM session_ids
    GROUP BY customer_id, session_id
) t
GROUP BY customer_id;
``

In [0]:
from pyspark.sql.functions import col, lag, sum, count, max, min
from pyspark.sql.window import Window

w = Window.partitionBy("customer_id").orderBy("event_time")

df = web_logs_df.withColumn(
    "prev_event",
    lag(col("event_time")).over(w)
).withColumn(
    "new_session",
    ((col("event_time").cast("long") -
      col("prev_event").cast("long")) / 60 > 30).cast("int")
)

sessions_df = df.withColumn(
    "session_id",
    sum("new_session").over(w)
)

agg_df = (
    sessions_df
    .groupBy("customer_id", "session_id")
    .agg(
        count("*").alias("page_count"),
        ((max("event_time").cast("long") -
          min("event_time").cast("long")) / 60).alias("session_duration")
    )
)

result_df = (
    agg_df
    .groupBy("customer_id")
    .agg(
        count("session_id").alias("total_sessions"),
        sum("page_count") / count("session_id"),
        sum("session_duration") / count("session_id")
    )
)

result_df.show()


+-----------+--------------+-------------------------------------+-------------------------------------------+
|customer_id|total_sessions|(sum(page_count) / count(session_id))|(sum(session_duration) / count(session_id))|
+-----------+--------------+-------------------------------------+-------------------------------------------+
|          1|            13|                   1.0769230769230769|                                        0.0|
|          2|             9|                   1.1111111111111112|                                        0.0|
|          3|            11|                   1.0909090909090908|                                        0.0|
|          4|            16|                               1.0625|                                        0.0|
|          5|            11|                   1.0909090909090908|                                        0.0|
|          6|            22|                   1.0454545454545454|                                        0.0|
|

## Q10. Which pairs of products are most often bought together in the same order? Output the top 20 pairs with co-occurrence counts and the lift ( P(A∩B) / (P(A)·P(B)) ).

SELECT
  p1.product_id AS product_a,
  p2.product_id AS product_b,
  COUNT(*) AS co_occurrence
FROM order_items p1
JOIN order_items p2
  ON p1.order_id = p2.order_id
 AND p1.product_id < p2.product_id
GROUP BY p1.product_id, p2.product_id
ORDER BY co_occurrence DESC
FETCH FIRST 20 ROWS ONLY;
``

In [0]:
from pyspark.sql.functions import col, count

pairs_df = (
    order_items_df.alias("a")
    .join(
        order_items_df.alias("b"),
        (col("a.order_id") == col("b.order_id")) &
        (col("a.product_id") < col("b.product_id"))
    )
    .groupBy(
        col("a.product_id").alias("product_a"),
        col("b.product_id").alias("product_b")
    )
    .agg(count("*").alias("co_occurrence"))
    .orderBy(col("co_occurrence").desc())
    .limit(20)
)

pairs_df.show()


+---------+---------+-------------+
|product_a|product_b|co_occurrence|
+---------+---------+-------------+
|        5|        9|           11|
|       52|       84|           10|
|       57|       76|           10|
|       41|       54|           10|
|       18|       79|           10|
|       26|       33|            9|
|       32|       34|            9|
|       31|       53|            9|
|        8|       36|            9|
|       56|       86|            9|
|        5|       17|            9|
|       11|       75|            9|
|       18|       60|            9|
|       43|       68|            9|
|        7|       51|            9|
|        6|       56|            9|
|       42|       82|            9|
|       30|       41|            9|
|       48|       62|            8|
|       48|       65|            8|
+---------+---------+-------------+



## Q11. For each (category, month) compute revenue, MoM absolute change, and MoM % change. Flag months where growth exceeded +20% or dropped below -20%.
WITH monthly_revenue AS (
    SELECT
        p.category,
        DATE_TRUNC(month, o.order_date) AS month,
        SUM(oi.line_total) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.category, DATE_TRUNC(month, o.order_date)
),
lagged AS (
    SELECT
        category,
        month,
        revenue,
        LAG(revenue) OVER (
            PARTITION BY category
            ORDER BY month
        ) AS prev_revenue
    FROM monthly_revenue
)
SELECT
    category,
    month,
    revenue,
    revenue - prev_revenue AS mom_change,
    (revenue - prev_revenue) / prev_revenue * 100 AS mom_pct_change
FROM lagged
WHERE prev_revenue IS NOT NULL;


In [0]:
from pyspark.sql.functions import col, sum, date_trunc, lag
from pyspark.sql.window import Window

monthly_df = (
    orders_df
    .join(order_items_df, orders_df.order_id == order_items_df.order_id)
    .join(products_df, order_items_df.product_id == products_df.product_id)
    .groupBy(
        products_df.category,
        date_trunc("month", orders_df.order_date).alias("month")
    )
    .agg(sum(order_items_df.line_total).alias("revenue"))
)

w = Window.partitionBy("category").orderBy("month")

result_df = (
    monthly_df
    .withColumn("prev_revenue", lag(col("revenue")).over(w))
    .filter(col("prev_revenue").isNotNull())
    .withColumn("mom_change", col("revenue") - col("prev_revenue"))
    .withColumn("mom_pct_change", (col("revenue") - col("prev_revenue")) / col("prev_revenue") * 100)
)

result_df.show()


+--------+-------------------+------------------+------------------+-------------------+-------------------+
|category|              month|           revenue|      prev_revenue|         mom_change|     mom_pct_change|
+--------+-------------------+------------------+------------------+-------------------+-------------------+
|   Books|2023-02-01 00:00:00| 5054725.049999998| 4923622.980000001| 131102.06999999657|  2.662715454301429|
|   Books|2023-03-01 00:00:00| 6452823.199999999| 5054725.049999998| 1398098.1500000013| 27.659232424521328|
|   Books|2023-04-01 00:00:00|        5058121.82| 6452823.199999999| -1394701.379999999|-21.613816724437747|
|   Books|2023-05-01 00:00:00| 5445424.510000002|        5058121.82| 387302.69000000134|   7.65704551576026|
|   Books|2023-06-01 00:00:00| 4771540.320000001| 5445424.510000002| -673884.1900000004| -12.37523702261369|
|   Books|2023-07-01 00:00:00| 4202458.160000001| 4771540.320000001| -569082.1600000001|-11.926592291689992|
|   Books|2023-08-0

## Q12. For every customer who has ≥ 3 orders, find their first-ever order and latest order, and compute the days between them and total orders in that span.

SELECT customer_id,
       MIN(order_date) AS first_order,
       MAX(order_date) AS last_order,
       DATEDIFF(MAX(order_date), MIN(order_date)) AS days_between,
       COUNT(order_id) AS total_orders
FROM orders
GROUP BY customer_id
HAVING COUNT(order_id) >= 3;

In [0]:
from pyspark.sql.functions import min, max, count, datediff, col

result_df = (
    orders_df
    .groupBy(col("customer_id"))
    .agg(
        min(col("order_date")).alias("first_order"),
        max(col("order_date")).alias("last_order"),
        count(col("order_id")).alias("total_orders")
    )
    .filter(col("total_orders") >= 3)
    .withColumn(
        "days_between",
        datediff(col("last_order"), col("first_order"))
    )
)

result_df.show()


+-----------+-------------------+-------------------+------------+------------+
|customer_id|        first_order|         last_order|total_orders|days_between|
+-----------+-------------------+-------------------+------------+------------+
|        496|2024-01-29 21:12:00|2025-02-09 10:50:00|           3|         377|
|        471|2023-02-25 22:33:00|2025-04-01 10:03:00|           9|         766|
|         70|2023-06-11 23:27:00|2025-02-07 13:58:00|           6|         607|
|        161|2023-05-03 04:11:00|2024-09-28 15:40:00|           6|         514|
|        275|2023-01-20 06:36:00|2025-04-21 07:24:00|           7|         822|
|        412|2023-04-04 04:30:00|2025-04-06 00:10:00|          10|         733|
|        340|2023-05-14 07:18:00|2025-05-13 11:35:00|           4|         730|
|        452|2023-06-14 11:47:00|2024-07-17 15:03:00|           6|         399|
|        434|2023-01-28 08:04:00|2025-05-21 02:36:00|          11|         844|
|        439|2023-08-15 20:14:00|2025-02

## Q13. For each (brand, category), compute: orders sold, orders returned, return rate %, and revenue lost from returns. Keep only combinations with ≥ 50 sold orders and return rate > the overall return rate.

WITH sold AS (
    SELECT
        p.brand,
        p.category,
        COUNT(DISTINCT oi.order_id) AS sold_orders,
        SUM(oi.line_total) AS revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.brand, p.category
),
returned AS (
    SELECT
        p.brand,
        p.category,
        COUNT(DISTINCT r.order_id) AS returned_orders,
        SUM(oi.line_total) AS revenue_lost
    FROM returns r
    JOIN order_items oi ON r.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.brand, p.category
),
combined AS (
    SELECT
        s.brand,
        s.category,
        s.sold_orders,
        COALESCE(r.returned_orders, 0) AS returned_orders,
        COALESCE(r.revenue_lost, 0) AS revenue_lost
    FROM sold s
    LEFT JOIN returned r
        ON s.brand = r.brand AND s.category = r.category
),
overall AS (
    SELECT
        SUM(returned_orders) / SUM(sold_orders) AS overall_return_rate
    FROM combined
)
SELECT
    brand,
    category,
    sold_orders,
    returned_orders,
    returned_orders * 100.0 / sold_orders AS return_rate,
    revenue_lost
FROM combined
WHERE sold_orders >= 50
AND returned_orders * 1.0 / sold_orders >
    (SELECT overall_return_rate FROM overall);



In [0]:
from pyspark.sql.functions import col, countDistinct, sum, coalesce

sold_df = (
    order_items_df
    .join(products_df, order_items_df.product_id == products_df.product_id)
    .groupBy(products_df.brand, products_df.category)
    .agg(
        countDistinct(order_items_df.order_id).alias("sold_orders"),
        sum(order_items_df.line_total).alias("revenue")
    )
)

returned_df = (
    returns_df
    .join(order_items_df, returns_df.order_id == order_items_df.order_id)
    .join(products_df, order_items_df.product_id == products_df.product_id)
    .groupBy(products_df.brand, products_df.category)
    .agg(
        countDistinct(returns_df.order_id).alias("returned_orders"),
        sum(order_items_df.line_total).alias("revenue_lost")
    )
)

combined_df = (
    sold_df
    .join(returned_df, ["brand", "category"], "left")
    .withColumn("returned_orders", coalesce(col("returned_orders"), col("sold_orders") * 0))
    .withColumn("revenue_lost", coalesce(col("revenue_lost"), col("sold_orders") * 0))
)

overall_rate = (
    combined_df
    .select(sum(col("returned_orders")) / sum(col("sold_orders")))
    .collect()[0][0]
)

result_df = (
    combined_df
    .filter(col("sold_orders") >= 50)
    .withColumn("return_rate", col("returned_orders") / col("sold_orders"))
    .filter(col("return_rate") > overall_rate)
    .select(
        col("brand"),
        col("category"),
        col("sold_orders"),
        col("returned_orders"),
        col("return_rate") * 100,
        col("revenue_lost")
    )
)

result_df.show()


+------+-----------+-----------+---------------+-------------------+------------------+
| brand|   category|sold_orders|returned_orders|(return_rate * 100)|      revenue_lost|
+------+-----------+-----------+---------------+-------------------+------------------+
|BrandA|Electronics|        206|             17|   8.25242718446602|2049511.4300000002|
|BrandA|      Books|        295|             24|  8.135593220338983|         955842.12|
|BrandB|      Books|        254|             29|  11.41732283464567| 819233.8899999997|
|BrandE|Electronics|        408|             32| 7.8431372549019605|2044691.1400000001|
|BrandD|      Books|        571|             44|  7.705779334500876|        3832977.05|
|BrandE|    Grocery|        204|             18|  8.823529411764707| 806917.9200000002|
|BrandE|      Books|        309|             25|  8.090614886731391|2943724.4900000007|
|BrandB|Electronics|        478|             36|  7.531380753138076|4608206.8599999985|
|BrandA|       Home|        384|

## Q14. List employees whose salary is above their own department's average. Show: name, dept, salary, dept_avg, diff, and rank within dept.

WITH dept_avg AS (
    SELECT
        department,
        AVG(salary) AS dept_avg_salary
    FROM employees
    GROUP BY department
)
SELECT
    e.emp_name,
    e.department,
    e.salary,
    d.dept_avg_salary,
    e.salary - d.dept_avg_salary AS diff
FROM employees e
JOIN dept_avg d
  ON e.department = d.department
WHERE e.salary > d.dept_avg_salary;

In [0]:
from pyspark.sql.functions import col, avg

dept_avg_df = (
    employees_df
    .groupBy(col("department"))
    .agg(avg(col("salary")).alias("dept_avg_salary"))
)

result_df = (
    employees_df
    .join(dept_avg_df, "department")
    .filter(col("salary") > col("dept_avg_salary"))
    .select(
        col("emp_name"),
        col("department"),
        col("salary"),
        col("dept_avg_salary"),
        col("salary") - col("dept_avg_salary")
    )
)

result_df.show()


+--------+-----------+------+-----------------+--------------------------+
|emp_name| department|salary|  dept_avg_salary|(salary - dept_avg_salary)|
+--------+-----------+------+-----------------+--------------------------+
|    VP_2|    Finance|176000|          77063.0|                   98937.0|
|    VP_3|         HR|174000| 71256.4074074074|         102743.5925925926|
|    VP_4| Operations|172000|          77069.1|                   94930.9|
|    VP_5|  Marketing|170000|72059.88888888889|         97940.11111111111|
|   Mgr_6|         HR| 98485| 71256.4074074074|          27228.5925925926|
|   Mgr_7|    Finance|121511|          77063.0|                   44448.0|
|   Mgr_8| Operations|109386|          77069.1|        32316.899999999994|
|   Mgr_9|         HR|119535| 71256.4074074074|          48278.5925925926|
|  Mgr_10|Engineering|129088|          70498.6|        58589.399999999994|
|  Mgr_11|  Marketing|101386|72059.88888888889|         29326.11111111111|
|  Mgr_12|    Finance|127

##Q15. Standard e-commerce funnel: /home → /search → /product → /cart → /checkout → /thankyou. For each customer, find the deepest stage they reached in each day's first session. Report overall funnel drop-off counts and conversion % at each step.


WITH ordered AS (
    SELECT
        customer_id,
        event_time,
        page,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id, CAST(event_time AS DATE)
            ORDER BY event_time
        ) AS rn
    FROM web_logs
),
funnel AS (
    SELECT
        customer_id,
        CAST(event_time AS DATE) AS event_date,
        MAX(CASE WHEN page = /home THEN 1 ELSE 0 END) AS home,
        MAX(CASE WHEN page = /search THEN 1 ELSE 0 END) AS search,
        MAX(CASE WHEN page = /product THEN 1 ELSE 0 END) AS product,
        MAX(CASE WHEN page = /cart THEN 1 ELSE 0 END) AS cart,
        MAX(CASE WHEN page = /checkout THEN 1 ELSE 0 END) AS checkout,
        MAX(CASE WHEN page = /thankyou THEN 1 ELSE 0 END) AS thankyou
    FROM ordered
    WHERE rn = 1
    GROUP BY customer_id, CAST(event_time AS DATE)
)
SELECT
    SUM(home) AS home_users,
    SUM(search) AS search_users,
    SUM(product) AS product_users,
    SUM(cart) AS cart_users,
    SUM(checkout) AS checkout_users,
    SUM(thankyou) AS thankyou_users
FROM funnel;
``

In [0]:
from pyspark.sql.functions import col, row_number, max
from pyspark.sql.window import Window

w = Window.partitionBy("customer_id", col("event_time").cast("date")) \
          .orderBy(col("event_time"))

df = web_logs_df.withColumn("rn", row_number().over(w))

funnel_df = (
    df.filter(col("rn") == 1)
    .groupBy(col("customer_id"), col("event_time").cast("date").alias("event_date"))
    .agg(
        max((col("page") == "/home").cast("int")).alias("home"),
        max((col("page") == "/search").cast("int")).alias("search"),
        max((col("page") == "/product").cast("int")).alias("product"),
        max((col("page") == "/cart").cast("int")).alias("cart"),
        max((col("page") == "/checkout").cast("int")).alias("checkout"),
        max((col("page") == "/thankyou").cast("int")).alias("thankyou")
    )
)

result_df = (
    funnel_df
    .agg(
        max("home"),
        max("search"),
        max("product"),
        max("cart"),
        max("checkout"),
        max("thankyou")
    )
)

result_df.show()

+---------+-----------+------------+---------+-------------+-------------+
|max(home)|max(search)|max(product)|max(cart)|max(checkout)|max(thankyou)|
+---------+-----------+------------+---------+-------------+-------------+
|        1|          1|           1|        1|            1|            1|
+---------+-----------+------------+---------+-------------+-------------+



## Q16.  Compute RFM for every customer: R = days since last order, F = order count last 365 days, M = total spend last 365 days. Bucket each into quintiles (1..5) and assign an RFM_segment label: Champions / Loyal / At-Risk / Lost / New.

WITH base AS (
    SELECT
        c.customer_id,
        DATEDIFF(CURRENT_DATE, MAX(o.order_date)) AS recency,
        COUNT(DISTINCT o.order_id) AS frequency,
        SUM(oi.line_total) AS monetary
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    LEFT JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY c.customer_id
),
scored AS (
    SELECT
        customer_id,
        recency,
        frequency,
        monetary,
        NTILE(5) OVER (ORDER BY recency DESC) AS r_score,
        NTILE(5) OVER (ORDER BY frequency) AS f_score,
        NTILE(5) OVER (ORDER BY monetary) AS m_score
    FROM base
)
SELECT
    customer_id,
    r_score,
    f_score,
    m_score,
    r_score + f_score + m_score AS rfm_score
FROM scored;

In [0]:
from pyspark.sql.functions import col, max, countDistinct, sum, datediff, current_date, ntile
from pyspark.sql.window import Window

base_df = (
    customers_df
    .join(orders_df, customers_df.customer_id == orders_df.customer_id, "left")
    .join(order_items_df, orders_df.order_id == order_items_df.order_id, "left")
    .groupBy(customers_df.customer_id)
    .agg(
        datediff(current_date(), max(orders_df.order_date)).alias("recency"),
        countDistinct(orders_df.order_id).alias("frequency"),
        sum(order_items_df.line_total).alias("monetary")
    )
)

w_recency = Window.orderBy(col("recency").desc())
w_frequency = Window.orderBy(col("frequency"))
w_monetary = Window.orderBy(col("monetary"))

result_df = (
    base_df
    .withColumn("r_score", ntile(5).over(w_recency))
    .withColumn("f_score", ntile(5).over(w_frequency))
    .withColumn("m_score", ntile(5).over(w_monetary))
    .withColumn("rfm_score", col("r_score") + col("f_score") + col("m_score"))
)

result_df.show()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+-------+---------+------------------+-------+-------+-------+---------+
|customer_id|recency|frequency|          monetary|r_score|f_score|m_score|rfm_score|
+-----------+-------+---------+------------------+-------+-------+-------+---------+
|        173|    770|        1|          11786.19|      1|      1|      1|        3|
|        101|    754|        1|          79477.24|      1|      1|      1|        3|
|        317|    583|        1|          163479.4|      1|      1|      1|        3|
|         87|    833|        2|         165171.66|      1|      1|      1|        3|
|         82|    396|        3|         178528.82|      3|      1|      1|        5|
|        483|    354|        2|209549.24999999997|      4|      1|      1|        6|
|        461|    792|        2|         224676.88|      1|      1|      1|        3|
|        144|    548|        3|         240402.64|      1|      1|      1|        3|
|        236|    574|        2|269212.37999999995|      1|      1

## Q17. For each customer segment, show one row with columns: UPI %, CreditCard %, DebitCard %, NetBanking %, COD %, Wallet % (by order count).

WITH base AS (
    SELECT
        c.segment,
        o.payment
    FROM orders o
    JOIN customers c
        ON o.customer_id = c.customer_id
),
counts AS (
    SELECT
        segment,
        payment,
        COUNT(*) AS cnt
    FROM base
    GROUP BY segment, payment
),
totals AS (
    SELECT
        segment,
        SUM(cnt) AS total_cnt
    FROM counts
    GROUP BY segment
)
SELECT
    c.segment,
    SUM(CASE WHEN c.payment = UPI THEN c.cnt * 100.0 / t.total_cnt ELSE 0 END) AS upi_pct,
    SUM(CASE WHEN c.payment = CreditCard THEN c.cnt * 100.0 / t.total_cnt ELSE 0 END) AS creditcard_pct,
    SUM(CASE WHEN c.payment = DebitCard THEN c.cnt * 100.0 / t.total_cnt ELSE 0 END) AS debitcard_pct,
    SUM(CASE WHEN c.payment = NetBanking THEN c.cnt * 100.0 / t.total_cnt ELSE 0 END) AS netbanking_pct,
    SUM(CASE WHEN c.payment = COD THEN c.cnt * 100.0 / t.total_cnt ELSE 0 END) AS cod_pct,
    SUM(CASE WHEN c.payment = Wallet THEN c.cnt * 100.0 / t.total_cnt ELSE 0 END) AS wallet_pct
FROM counts c
JOIN totals t
    ON c.segment = t.segment
GROUP BY c.segment;


In [0]:
from pyspark.sql.functions import col, count, sum

base_df = (
    orders_df
    .join(customers_df, orders_df.customer_id == customers_df.customer_id)
    .select(customers_df.segment, orders_df.payment_mode)
)

counts_df = (
    base_df
    .groupBy(col("segment"), col("payment_mode"))
    .agg(count("*").alias("cnt"))
)

totals_df = (
    counts_df
    .groupBy(col("segment"))
    .agg(sum(col("cnt")).alias("total_cnt"))
)

result_df = (
    counts_df
    .join(totals_df, "segment")
    .withColumn("pct", col("cnt") * 100 / col("total_cnt"))
    .groupBy(col("segment"))
    .pivot("payment_mode")
    .agg(sum(col("pct")))
)

result_df.show()


+--------+------------------+------------------+------------------+------------------+------------------+------------------+
| segment|               COD|        CreditCard|         DebitCard|        NetBanking|               UPI|            Wallet|
+--------+------------------+------------------+------------------+------------------+------------------+------------------+
|    Gold|17.785630153121318|17.078916372202592| 16.72555948174323|17.785630153121318| 15.07656065959953|15.547703180212014|
| Premium|17.255434782608695|14.266304347826088| 14.53804347826087|20.108695652173914|17.527173913043477|16.304347826086957|
| Regular| 16.31419939577039|15.558912386706949|16.012084592145015| 17.97583081570997| 16.76737160120846|17.371601208459214|
|Platinum| 15.53784860557769|  15.0066401062417|16.865869853917662|17.131474103585656|18.326693227091635|17.131474103585656|
+--------+------------------+------------------+------------------+------------------+------------------+------------------+


##Q18. Flag potential duplicates — same customer, same total amount, placed within 10 minutes of each other. These usually indicate double-submits.

SELECT
    o1.order_id AS order_1,
    o2.order_id AS order_2,
    o1.customer_id,
    o1.order_date,
    o2.order_date,
    o1_total.total_amount
FROM orders o1
JOIN orders o2
  ON o1.customer_id = o2.customer_id
 AND o1.order_id < o2.order_id
JOIN (
    SELECT order_id, SUM(line_total) AS total_amount
    FROM order_items
    GROUP BY order_id
) o1_total
  ON o1.order_id = o1_total.order_id
JOIN (
    SELECT order_id, SUM(line_total) AS total_amount
    FROM order_items
    GROUP BY order_id
) o2_total
  ON o2.order_id = o2_total.order_id
WHERE o1_total.total_amount = o2_total.total_amount
AND ABS(TIMESTAMPDIFF(minute, o1.order_date, o2.order_date)) <= 10;


In [0]:
from pyspark.sql.functions import col, sum, abs

order_totals_df = (
    order_items_df
    .groupBy(col("order_id"))
    .agg(sum(col("line_total")).alias("total_amount"))
)

orders_with_total_df = (
    orders_df
    .join(order_totals_df, "order_id")
)

result_df = (
    orders_with_total_df.alias("a")
    .join(
        orders_with_total_df.alias("b"),
        (col("a.customer_id") == col("b.customer_id")) &
        (col("a.order_id") < col("b.order_id")) &
        (col("a.total_amount") == col("b.total_amount"))
    )
    .filter(
        abs(col("a.order_date").cast("long") -
            col("b.order_date").cast("long")) <= 600
    )
    .select(
        col("a.order_id"),
        col("b.order_id"),
        col("a.customer_id"),
        col("a.total_amount")
    )
)

result_df.show()

+--------+--------+-----------+------------+
|order_id|order_id|customer_id|total_amount|
+--------+--------+-----------+------------+
+--------+--------+-----------+------------+



##Q19. Sort customers by descending spend. Find the smallest set of customers who together account for ≥ 80% of total revenue (the classic Pareto / "whale" analysis).

WITH customer_revenue AS (
    SELECT
        o.customer_id,
        SUM(oi.line_total) AS revenue
    FROM orders o
    JOIN order_items oi
        ON o.order_id = oi.order_id
    GROUP BY o.customer_id
),
ordered AS (
    SELECT
        customer_id,
        revenue,
        SUM(revenue) OVER (ORDER BY revenue DESC) AS running_revenue,
        SUM(revenue) OVER () AS total_revenue
    FROM customer_revenue
)
SELECT
    customer_id,
    revenue,
    running_revenue,
    total_revenue,
    running_revenue / total_revenue * 100 AS cumulative_pct
FROM ordered
WHERE running_revenue / total_revenue <= 0.8
ORDER BY revenue DESC;

In [0]:
from pyspark.sql.functions import col, sum
from pyspark.sql.window import Window

customer_revenue_df = (
    orders_df
    .join(order_items_df, orders_df.order_id == order_items_df.order_id)
    .groupBy(orders_df.customer_id)
    .agg(sum(order_items_df.line_total).alias("revenue"))
)

total_revenue = (
    customer_revenue_df
    .select(sum(col("revenue")))
    .collect()[0][0]
)

w = Window.orderBy(col("revenue").desc())

result_df = (
    customer_revenue_df
    .withColumn("running_revenue", sum(col("revenue")).over(w))
    .withColumn("cumulative_pct", col("running_revenue") / total_revenue)
    .filter(col("cumulative_pct") <= 0.8)
)

result_df.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+------------------+--------------------+--------------------+
|customer_id|           revenue|     running_revenue|      cumulative_pct|
+-----------+------------------+--------------------+--------------------+
|        228|5140218.0200000005|  5140218.0200000005|0.006491916378682...|
|        359|        3856154.83|   8996372.850000001|0.011362105659022907|
|        124|3791457.3599999994|       1.278783021E7| 0.01615058428749594|
|        418|        3704824.15|1.6492654360000001E7|0.020829648188276765|
|        356|3698192.1999999993|2.0190846560000002E7|0.025500336167129753|
|        414|        3618794.25|2.3809640810000002E7| 0.03007074729974133|
|        444|3608492.8299999987|       2.741813364E7|0.034628148097584716|
|        410|3330252.1599999988|        3.07483858E7| 0.03883414061746002|
|        322|3318755.5299999993|       3.406714133E7|0.043025613293953914|
|        266|        3314983.82|       3.738212515E7| 0.04721232243204743|
|        463|3306867.8300

## Q20. Produce a single JSON-like audit object containing:

1.Orders where SUM(order_items.line_total) ≠ order-level total (mismatched totals)<BR>
2.Customers with orders but no entry in customers.csv (orphan FK)<BR>
3.Products never ordered (dead SKUs)<BR>
4.Employees whose manager_id doesn't exist in employees.csv<BR>
5.Duplicate order_ids or customer_ids<BR>
6.Future-dated orders (order_date > today)<BR>
7.Negative or zero line_total values<BR>


SELECT
    mismatched_orders,
    orphan_customers,
    dead_products,
    invalid_managers,
    duplicate_orders,
    future_orders,
    invalid_line_totals
FROM (
    SELECT
        COUNT(*) AS mismatched_orders
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.order_id
    HAVING SUM(oi.line_total) <> SUM(oi.line_total)
) a,
(
    SELECT COUNT(*) AS orphan_customers
    FROM orders o
    LEFT JOIN customers c ON o.customer_id = c.customer_id
    WHERE c.customer_id IS NULL
) b,
(
    SELECT COUNT(*) AS dead_products
    FROM products p
    LEFT JOIN order_items oi ON p.product_id = oi.product_id
    WHERE oi.product_id IS NULL
) c,
(
    SELECT COUNT(*) AS invalid_managers
    FROM employees e
    LEFT JOIN employees m ON e.manager_id = m.emp_id
    WHERE e.manager_id IS NOT NULL AND m.emp_id IS NULL
) d,
(
    SELECT COUNT(*) AS duplicate_orders
    FROM orders
    GROUP BY order_id
    HAVING COUNT(*) > 1
) e,
(
    SELECT COUNT(*) AS future_orders
    FROM orders
    WHERE order_date > CURRENT_DATE
) f,
(
    SELECT COUNT(*) AS invalid_line_totals
    FROM order_items
    WHERE line_total <= 0
) g;
``

In [0]:
from pyspark.sql.functions import col, sum, count

mismatched_orders = (
    orders_df
    .join(order_items_df, orders_df.order_id == order_items_df.order_id)
    .groupBy(orders_df.order_id)
    .agg(sum(order_items_df.line_total).alias("total"))
    .filter(col("total") != col("total"))
    .count()
)

orphan_customers = (
    orders_df
    .join(customers_df, orders_df.customer_id == customers_df.customer_id, "left")
    .filter(customers_df.customer_id.isNull())
    .count()
)

dead_products = (
    products_df
    .join(order_items_df, products_df.product_id == order_items_df.product_id, "left")
    .filter(order_items_df.product_id.isNull())
    .count()
)

invalid_managers = (
    employees_df.alias("e")
    .join(
        employees_df.alias("m"),
        col("e.manager_id") == col("m.emp_id"),
        "left"
    )
    .filter(col("e.manager_id").isNotNull() & col("m.emp_id").isNull())
    .count()
)

duplicate_orders = (
    orders_df
    .groupBy(col("order_id"))
    .count()
    .filter(col("count") > 1)
    .count()
)

future_orders = (
    orders_df
    .filter(col("order_date") > current_date())
    .count()
)

invalid_line_totals = (
    order_items_df
    .filter(col("line_total") <= 0)
    .count()
)

spark.createDataFrame(
    [(mismatched_orders, orphan_customers, dead_products, invalid_managers, duplicate_orders, future_orders, invalid_line_totals)],
    ["mismatched_orders","orphan_customers","dead_products","invalid_managers","duplicate_orders","future_orders","invalid_line_totals"]
).show()


+-----------------+----------------+-------------+----------------+----------------+-------------+-------------------+
|mismatched_orders|orphan_customers|dead_products|invalid_managers|duplicate_orders|future_orders|invalid_line_totals|
+-----------------+----------------+-------------+----------------+----------------+-------------+-------------------+
|                0|               0|            0|               0|               0|            0|                  0|
+-----------------+----------------+-------------+----------------+----------------+-------------+-------------------+

